In [213]:
import numpy as np
import pandas as pd

In [214]:
train=pd.read_csv(r"C:\Users\U.SIDDIQ ALIKHAN\Downloads\titanic\train.csv")
test=pd.read_csv(r"C:\Users\U.SIDDIQ ALIKHAN\Downloads\titanic\test.csv")

In [215]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [216]:
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [217]:
train.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [218]:
passenger_id=train["PassengerId"]
passenger_i1=test["PassengerId"]

In [219]:
test.isnull().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [220]:
test.drop("Cabin", axis=1, inplace=True)

In [221]:
train["Title"]=train["Name"].str.extract(r"([A-Za-z]+)\.", expand=False)
test["Title"]=test["Name"].str.extract(r"([A-Za-z]+)\.",expand=False)

In [222]:
rare = ["Lady","Countess","Capt","Col","Don","Dr","Major","Rev","Sir","Jonkheer","Dona"]

train["Title"] = train["Title"].replace(rare, "Rare")
test["Title"] = test["Title"].replace(rare, "Rare")

In [223]:
train["Title"] = train["Title"].fillna("Rare")
test["Title"] = test["Title"].fillna("Rare")

In [224]:
print(train["Title"].value_counts())

Title
Mr        517
Miss      182
Mrs       125
Master     40
Rare       23
Mlle        2
Mme         1
Ms          1
Name: count, dtype: int64


In [225]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

train["Title"] = le.fit_transform(train["Title"])
test["Title"] = le.transform(test["Title"])

In [226]:
print(train["Title"].isnull().sum())

0


In [227]:
train.drop("Name", axis=1, inplace=True)
test.drop("Name", axis=1, inplace=True)

In [228]:
#data exploratory.. which people survived the most
train.groupby("Sex")["Survived"].mean()

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

In [229]:
train.groupby("Pclass")["Survived"].mean()

Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64

In [230]:
train.groupby("Title")["Survived"].mean()

Title
0    0.575000
1    0.697802
2    1.000000
3    1.000000
4    0.156673
5    0.792000
6    1.000000
7    0.347826
Name: Survived, dtype: float64

In [231]:
#🔍 What this means
#Column	Meaning
#SibSp	siblings/spouse
#Parch	parents/children
#+1	the person themselves
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1

In [232]:
train.groupby("FamilySize")["Survived"].mean()

FamilySize
1     0.303538
2     0.552795
3     0.578431
4     0.724138
5     0.200000
6     0.136364
7     0.333333
8     0.000000
11    0.000000
Name: Survived, dtype: float64

In [233]:
train["FamilyGroup"] = train["FamilySize"]
test["FamilyGroup"] = test["FamilySize"]

In [234]:
train.loc[train["FamilyGroup"] == 1, "FamilyGroup"] = 0   # Alone
train.loc[(train["FamilyGroup"] >= 2) & (train["FamilyGroup"] <= 4), "FamilyGroup"] = 1  # Small family
train.loc[train["FamilyGroup"] >= 5, "FamilyGroup"] = 2   # Large family

test.loc[test["FamilyGroup"] == 1, "FamilyGroup"] = 0
test.loc[(test["FamilyGroup"] >= 2) & (test["FamilyGroup"] <= 4), "FamilyGroup"] = 1
test.loc[test["FamilyGroup"] >= 5, "FamilyGroup"] = 2

In [235]:
train["FamilyGroup"].value_counts()

FamilyGroup
0    537
1    292
2     62
Name: count, dtype: int64

In [236]:
test["FamilyGroup"].value_counts()

FamilyGroup
0    253
1    145
2     20
Name: count, dtype: int64

In [237]:
train["Sex"] = train["Sex"].map({"male": 0, "female": 1})
test["Sex"] = test["Sex"].map({"male": 0, "female": 1})

In [238]:
train["Age"] = train["Age"].fillna(train["Age"].median())
test["Age"] = test["Age"].fillna(test["Age"].median())

In [239]:
test["Fare"] = test["Fare"].fillna(test["Fare"].median())

In [240]:
train["Embarked"] = train["Embarked"].fillna(train["Embarked"].mode()[0])
test["Embarked"] = test["Embarked"].fillna(test["Embarked"].mode()[0])

In [241]:
train["Embarked"] = train["Embarked"].map({"S": 0, "C": 1, "Q": 2})
test["Embarked"] = test["Embarked"].map({"S": 0, "C": 1, "Q": 2})

In [242]:
X = train.drop(["Survived","Ticket", "PassengerId","Cabin"], axis=1)
y = train["Survived"]

In [243]:
X.isnull().sum()

Pclass         0
Sex            0
Age            0
SibSp          0
Parch          0
Fare           0
Embarked       0
Title          0
FamilySize     0
FamilyGroup    0
dtype: int64

In [244]:
test.isnull().sum()

PassengerId    0
Pclass         0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
Title          0
FamilySize     0
FamilyGroup    0
dtype: int64

In [245]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [246]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [247]:
pred = model.predict(X_val)

In [248]:
from sklearn.metrics import accuracy_score
print(accuracy_score(y_val, pred))

0.7988826815642458


In [249]:
model.fit(X, y)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [251]:
test_X = test.drop(["Ticket", "PassengerId"], axis=1)

predictions = model.predict(test_X)

In [252]:
test_X = test[X.columns]
predictions = model.predict(test_X)

In [254]:
ABDULWAHEEDKHAN = pd.DataFrame({
    "PassengerId": passenger_i1,
    "Survived": predictions
})

ABDULWAHEEDKHAN.to_csv("ABDULWAHEEDKHAN.csv", index=False)

In [256]:
ABDULWAHEEDKHAN.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
